# Phase 9 (pilot) — Robustness across models and prompt phrasing

Phase 4/5 established that, on **one fixed model** and **one fixed
phrasing** of the accessibility question, the LLM arm was perfectly
reliable across N=20 runs (PAR=1.0, Rank Stability=1.0). That is a
narrow result: it says nothing about whether perfect reliability is a
property of *this* model and *this* wording, or something more general.

This notebook runs a **small pilot** (N=5 per condition, not N=20) along
two separate axes, to check whether either one reintroduces the variance
that N=20 on the original condition did not show:

- **Axis A — alternative models**, same question, same `SYSTEM_HINTS`
  (the schema-safety instructions phase 4 spent four debugging rounds
  developing — see `claude/phase-4-llm-arm.md`). Held fixed deliberately:
  varying those too would just re-expose the *old* framework bugs, not
  test anything new about models or phrasing.
- **Axis B — alternative phrasings** of the same underlying request, same
  model phase 4 used (whatever `LLM_MODEL` is set to in `.env`).

**Escalation rule** (same discipline as `paper/PLAN.md`'s N=20 decision):
any condition that shows *any* deviation from PAR=1.0 / Rank Stability=1.0
/ success_rate=1.0 at this small N gets escalated to N=20 to characterize
it properly. A condition that stays clean at N=5 is reported as "stable at
N=5, not yet confirmed at N=20" — a 5-run pilot has real limits on its
power to detect low-frequency variance, and this notebook does not
overclaim past what N=5 can actually show.

**Important — execution status of this notebook.** Every helper function
below is copied from, or checked directly against, `smart_spatial_system`'s
real source (`orchestrator/planning/llm_spec_generator.py`) and every cell
was syntax-checked with `compile()` before delivery — but the actual LLM
API calls have **not** been made yet: this session has no AvalAI API key
and no route to install `smart_spatial_system`'s own dependencies. Fill in
the model IDs below, run the smoke-test cell first, then the full loop —
the real results should be re-verified the same way every previous phase's
notebook was (re-stage the executed copy and check it end to end).

## Setup

In [ ]:
import json
import os
import statistics
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from scipy.stats import spearmanr

from orchestrator.capability_registry import CapabilityRegistry
from orchestrator.planning.dag_executor import DagExecutor
from orchestrator.planning.planner import DeterministicPlanner
from orchestrator.planning.llm_spec_generator import (
    LLMQuerySpecGenerator,
    LLMSpecGenerationError,
    OpenAICompatibleLLMClient,
    normalize_llm_query_spec_for_planning,
    query_spec_to_dict,
)

load_dotenv("../.env")

PROCESSED = Path("../data/processed")
RESULTS = Path("../results")
PILOT_DIR = RESULTS / "robustness_pilot"
PILOT_DIR.mkdir(parents=True, exist_ok=True)

print("model env var set:", bool(os.environ.get("LLM_MODEL")))
print("an API key is set:", any(os.environ.get(k) for k in ("LLM_API_KEY", "AVALAI_API_KEY", "OPENAI_API_KEY")))


## Load phase 2/3/4 outputs, and phase 4's own `RAW_QUERY` / `SYSTEM_HINTS` with zero drift

Rather than copy-pasting phase 4's long `SYSTEM_HINTS` string a second
time (real risk of the two copies silently drifting apart over edits),
this reads `03_llm_arm.ipynb`'s own cells and executes them into a
throwaway namespace to pull the exact same objects out. If phase 4's
notebook is ever revised, this picks up the change automatically instead
of quietly comparing against a stale copy.

In [ ]:
def load_geojson(name):
    with open(PROCESSED / f"{name}.geojson") as f:
        return json.load(f)

sites_geojson = load_geojson("districts")
amenity_geojson = {
    "metro": load_geojson("metro"),
    "schools": load_geojson("schools"),
    "parks": load_geojson("parks"),
}
initial_inputs = {"sites": sites_geojson, **amenity_geojson}

rule_based_ranking_df = pd.read_csv(RESULTS / "rule_based_ranking.csv")
with open(RESULTS / "rule_based_query_spec.json") as f:
    rb_spec = json.load(f)

baseline_llm_run_files = sorted((RESULTS / "llm_runs").glob("run_*.json"))
baseline_llm_runs = [json.loads(f.read_text()) for f in baseline_llm_run_files]
assert len(baseline_llm_runs) == 20, (
    f"expected phase 4's 20 confirmed-clean baseline runs in results/llm_runs/, found {len(baseline_llm_runs)} - "
    "this pilot builds on top of that data, run phase 4 first if this fails."
)
assert all(r["generation_success"] and r["execution_success"] and not r["degenerate_ranking"] for r in baseline_llm_runs), \
    "phase 4's baseline runs are not all clean - re-check phase 4 before running this pilot"


def load_phase4_definitions():
    """Execute notebooks/03_llm_arm.ipynb's own setup cells into a throwaway
    namespace and pull out RAW_QUERY, SYSTEM_HINTS, TEMPERATURE - guarantees
    this pilot varies EXACTLY what phase 4 held fixed, with no copy-paste
    drift. Cell selection is by a short, distinctive substring of each
    cell's own source, not by a fragile cell index."""
    nb = json.loads(Path("03_llm_arm.ipynb").read_text())
    ns = {
        "OpenAICompatibleLLMClient": OpenAICompatibleLLMClient,
        "LLMQuerySpecGenerator": LLMQuerySpecGenerator,
        "os": os,
        "Path": Path,
    }
    found = {"query_cell": False, "hints_cell": False}
    for cell in nb["cells"]:
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell["source"])
        if 'RAW_QUERY = "Rank' in src:
            exec(src, ns)
            found["query_cell"] = True
        elif src.startswith("llm_client = OpenAICompatibleLLMClient()"):
            exec(src, ns)
            found["hints_cell"] = True
    if not (found["query_cell"] and found["hints_cell"]):
        raise RuntimeError(f"could not find phase 4's RAW_QUERY/SYSTEM_HINTS cells - found={found}")
    return ns["RAW_QUERY"], ns["SYSTEM_HINTS"], ns["TEMPERATURE"]


BASE_RAW_QUERY, SYSTEM_HINTS, TEMPERATURE = load_phase4_definitions()
print(f"loaded phase 4's BASE_RAW_QUERY ({len(BASE_RAW_QUERY)} chars) and SYSTEM_HINTS ({len(SYSTEM_HINTS)} chars) with zero drift")
print(f"BASE_RAW_QUERY = {BASE_RAW_QUERY!r}")
print(f"temperature = {TEMPERATURE}")


## First real run's findings, and one deliberate harness fix

The first real run of this notebook (2026-09-16) surfaced two genuine
findings, not code bugs in this notebook itself:

1. **A real, root-caused execution bug on `gpt-4o`.** All 5 generation
   calls succeeded and produced a plan with the same 9-op shape as the
   baseline - but all 5 executions failed with
   `rank_features() got an unexpected keyword argument 'ranking_field'`.
   Checked directly against `smart_spatial_system`'s own source
   (`plugins/feature_scoring.py`): the real parameter is `rank_field`
   (default `rank`) - there is no `ranking_field`. `SYSTEM_HINTS` (phase
   4's, loaded above with zero drift) documents `score_features`'s schema
   in exhaustive detail (six numbered requirements) but says **nothing**
   about `rank_features`'s parameter names. `gpt-4o-mini` (the baseline)
   apparently never needed that guidance - it always guesses right or
   omits the parameter - but `gpt-4o` guessed a plausible-sounding wrong
   name. This is the same *category* of failure phase 4 hit six times
   with `score_features` (see `claude/phase-4-llm-arm.md`), just in a
   part of the schema phase 4 never happened to need spelled out.

   **Deliberately NOT editing phase 4's own `SYSTEM_HINTS`** - it's
   quoted verbatim in `paper/paper.md`'s Methods section and its N=20
   result is already published; retroactively changing it would break
   that citation. Instead, this notebook defines its own
   `EXTENDED_SYSTEM_HINTS` below (phase 4's hints plus one more numbered
   requirement covering `rank_features`) and uses that for every pilot
   condition (both axes) from here on - a disclosed, one-time
   completeness fix to the harness, not a change to what's being tested.
   `gpt-4o`'s original 5 runs (0/5 execution success, under the
   *original* hints) are preserved on disk under
   `model_frontier_same_vendor/` exactly as they came out - a real,
   useful data point on hint completeness in its own right - and get
   re-run as a separate condition below under the extended hints for a
   fair comparison.

2. **3 of my 4 non-OpenAI model-id guesses are dead on AvalAI right now**
   (`gpt-3.5-turbo` and `gemini-2.0-flash`: HTTP 404 "model deprecated";
   `llama-3.1-8b-instruct`: HTTP 404 "resource not found"). One
   (`claude-3-5-haiku-20241022`) WAS recognized - AvalAI resolved it to
   `anthropic.claude-3-5-haiku-20241022-v1:0` internally - but that
   resolved model is *also* now deprecated. I have no way to query
   AvalAI's live catalog from here (their docs site blocks automated
   fetches, and I have no AvalAI key) - **fill in current ids from your
   own AvalAI dashboard/model picker** for the 4 marked `# NEEDS UPDATE`
   below, then re-run the smoke test.

## Final result (after the second and third real runs, 2026-09-16)

**7 of 9 conditions came back clean; 2 of 9 are accepted as blocked and
the pilot is now considered final** - the user has no further AvalAI
model ids to try for the two still-broken slots, so rather than keep
guessing, this pilot stops here and reports the honest result.

**Clean (PAR = 1.0, Rank Stability = 1.0, ρ = 0.998966, N = 5 each):**
`gpt-4o` (`model_frontier_same_vendor`, fixed by `EXTENDED_SYSTEM_HINTS`
above), `grok-3-mini-fast-beta` (`model_midtier_other_vendor`),
`grok-3-fast` (`model_small_open_weight`), and all four prompt
paraphrasings (`prompt_casual`/`formal`/`reordered`/`distractor` - the
`formal`/`distractor` PDFOut bug from the first run is also fixed by the
extended hints).

**Still blocked, for two distinct reasons:**

- `model_weak_legacy` (tried as `qwen3-235b-a22b-fp8-tput`): every one of
  5 generation calls failed with the same AvalAI `HTTP 400
  invalid_request` - this id is rejected outright, not a model behavior
  issue.
- `model_frontier_other_vendor` (tried as `qwen3.8-2.4t-a95b`): a genuinely
  mixed, more interesting failure. 3/5 calls timed out at 60s; the other
  2/5 *did* generate a plan, with the identical 9-op shape as every other
  condition (`PAR = 1.0` on the 2 that generated) - but both then failed
  *execution* with `score_features() got an unexpected keyword argument
  'score_field'`. Checked directly against `smart_spatial_system`'s real
  signature (`plugins/feature_scoring.py`): `score_features` has no
  `score_field` parameter at all (it takes `output_field` instead) -
  `score_field` is only valid on `rank_features`. This model applied the
  `rank_features` parameter name shown in `EXTENDED_SYSTEM_HINTS` to
  *both* ops, in the opposite direction from the original `gpt-4o` bug
  above. **A third, independent instance of the same harness-completeness
  lesson** (Section 6.1 of the paper): the hint fix that made `gpt-4o`
  reliable created a fresh, model-specific ambiguity for a different
  model, rather than closing the gap once for everyone. Worth keeping in
  the paper as a concrete illustration, even though this slot is not
  going to be resolved with a working model id.

**A real notebook bug this surfaced, fixed below:** the manifest- and
metrics-computation cells previously built their tables from `all_records`
in memory. When this loop was re-run for only the two still-broken
conditions above (an interrupted kernel session), `all_records` held only
*those* conditions' records - so recomputing metrics from memory alone
would have silently overwritten the six untouched, already-correct
conditions' results with blanks, even though their `run_*.json` files
were sitting on disk the whole time. Both cells now read every
condition's records directly from disk instead (see `load_condition_records`
below) - correct regardless of which subset of conditions the loop above
was actually re-run for.

In [ ]:
# One additional schema requirement SYSTEM_HINTS (phase 4's, loaded above)
# never needed: rank_features's real parameter name, confirmed against
# smart_spatial_system's own source (plugins/feature_scoring.py) - gpt-4o's
# first pilot run guessed the plausible-but-wrong "ranking_field" instead.
# Deliberately kept SEPARATE from phase 4's own SYSTEM_HINTS (see markdown
# above) - used for every condition in THIS notebook only.
RANK_FIELD_ADDENDUM = (
    " (7) whichever operation ranks sites by their score, set its parameter "
    "that names the rank/order output field explicitly to 'rank_field' "
    "(default value: the string rank) - not 'ranking_field' or any other "
    "name - and set 'score_field' to whatever field name you actually used "
    "for the combined score. Do not add a report/PDF-generation step unless "
    "the request explicitly asks for a document/report artifact - a plain "
    "ranking answer is a ranked feature list, not a rendered document."
)
EXTENDED_SYSTEM_HINTS = SYSTEM_HINTS + RANK_FIELD_ADDENDUM
print(f"EXTENDED_SYSTEM_HINTS: {len(SYSTEM_HINTS)} base chars + {len(RANK_FIELD_ADDENDUM)} added chars")


## Pilot configuration

**Model choice (Axis A).** I picked these 5 myself, spanning vendor and
capability tier, per your request. Two are OpenAI ids in the exact same
bare form your `.env`'s baseline (`gpt-4o-mini`) already uses successfully
against AvalAI, so those two are high-confidence. The other three
(Gemini/Claude/Llama) are my best-effort standard id strings for each
provider - I could **not** verify AvalAI's own model catalog directly
(their docs site refused automated fetching, and I have no AvalAI key to
query it myself), so treat those three as a starting guess, not a
confirmed list. That is exactly what the smoke-test cell below is for:
it costs one call per condition and will tell you immediately which ids
AvalAI rejects, cheaply, before the full N=5 loop spends more calls on a
typo - fix any `FAILED` line there (usually: swap in whatever exact
string your AvalAI dashboard/model-picker shows for that provider) and
re-run the smoke test before continuing.
The baseline model (whatever `LLM_MODEL` is set to in `.env` — already
characterized at N=20 in `results/llm_runs/`) is intentionally **not**
repeated here; it is the reference row both tables below compare against.

In [ ]:
N_PILOT = 5  # small pilot, not N=20 - see the escalation rule in the top markdown cell

# ---- Axis A: alternative models, same question (BASE_RAW_QUERY), same EXTENDED_SYSTEM_HINTS ----
# Updated from the first real run's results (see markdown above):
MODEL_IDS = {
    "frontier_same_vendor": "gpt-4o",  # CONFIRMED working on AvalAI - generation succeeded 5/5 under the ORIGINAL hints; re-run here under EXTENDED_SYSTEM_HINTS to test the rank_field fix
    # The 4 below all failed on AvalAI in the first run (see markdown above for the exact error each one returned).
    # Fill in a CURRENT id from your own AvalAI dashboard/model picker for each, then re-run the smoke test.
    "weak_legacy": "gpt-3.5-turbo",                      # NEEDS UPDATE - AvalAI: "model gpt-3.5-turbo has been deprecated"
    "frontier_other_vendor": "gemini-2.0-flash",         # NEEDS UPDATE - AvalAI: "model gemini-2.0-flash has been deprecated"
    "midtier_other_vendor": "claude-3-5-haiku-20241022", # NEEDS UPDATE - AvalAI resolved this to anthropic.claude-3-5-haiku-20241022-v1:0, which is ALSO "deprecated" - try a newer Claude id (e.g. a current Haiku/Sonnet snapshot)
    "small_open_weight": "llama-3.1-8b-instruct",        # NEEDS UPDATE - AvalAI: "Requested resource llama-3.1-8b-instruct does not exist"
}

# ---- Axis B: alternative phrasings of the SAME request, baseline model (env LLM_MODEL) ----
# Semantically equivalent to BASE_RAW_QUERY - none of these state the amenity
# weights/max_distance_m numbers either, for the same reason phase 4's didn't.
PROMPT_VARIANTS = {
    "casual": (
        "Which Viennese districts are easiest to get around without a car - "
        "close to metro, schools, and parks? Rank all 23, best to worst."
    ),
    "formal": (
        "Compute an accessibility ranking of all 23 administrative districts of "
        "Vienna with respect to proximity to public transit (metro stations), "
        "educational facilities (schools), and green space (parks)."
    ),
    "reordered": (
        "Considering parks, schools, and metro stations, order Vienna's 23 "
        "municipal districts from most to least accessible."
    ),
    "distractor": (
        "I'm putting together a relocation report for a client moving to Vienna "
        "with two school-age kids. As part of that, can you rank the city's 23 "
        "districts by how accessible they are to metro stations, schools, and parks?"
    ),
}

print(f"{len(MODEL_IDS)} model conditions, {len(PROMPT_VARIANTS)} prompt conditions, N_PILOT={N_PILOT} each")
print(f"planned new API calls: {(len(MODEL_IDS) + len(PROMPT_VARIANTS)) * N_PILOT} (plus {len(MODEL_IDS) + len(PROMPT_VARIANTS)} smoke-test calls)")


## Execution/scoring helpers (unchanged from phase 4 — same functions, same source)

In [ ]:
registry = CapabilityRegistry.from_plugin_modules(tolerant=True)

def run_pipeline(query_spec):
    plan = DeterministicPlanner().build(query_spec)
    return DagExecutor(lambda name: registry.resolve(name).callable).execute(
        plan, initial_inputs=initial_inputs,
    )

def vector_out_to_dataframe(vector_out):
    features = getattr(vector_out, "features", None)
    if features is None:
        raise TypeError(f"expected a VectorOut-like object with .features, got {type(vector_out)}")
    return pd.DataFrame([f.get("properties", {}) if isinstance(f, dict) else {} for f in features])

def plan_score_and_rank_fields(query_spec, default_score_field="accessibility_score", default_rank_field="rank"):
    score_field, rank_field = default_score_field, default_rank_field
    for op in query_spec.operations:
        params = getattr(op, "params", {}) or {}
        if "score_field" in params:
            score_field = params["score_field"]
        if "rank_field" in params:
            rank_field = params["rank_field"]
    return score_field, rank_field

def extract_ranking(result, query_spec):
    report = result.outputs.get("sites_report")
    if report is not None and hasattr(report, "table"):
        df = pd.DataFrame(report.table["rows"])
    else:
        terminal_name = query_spec.operations[-1].output
        terminal = result.outputs.get(terminal_name)
        if terminal is None:
            raise KeyError(
                f"neither 'sites_report' nor the plan's terminal output {terminal_name!r} "
                f"found in result.outputs (keys: {list(result.outputs.keys())})"
            )
        df = vector_out_to_dataframe(terminal)

    score_field, rank_field = plan_score_and_rank_fields(query_spec)
    if rank_field not in df.columns:
        raise KeyError(f"expected rank column {rank_field!r} not in columns {list(df.columns)}")
    if score_field not in df.columns:
        score_like = [c for c in df.columns if c == "score" or (c.endswith("_score") and c != rank_field)]
        if len(score_like) == 1:
            df = df.rename(columns={score_like[0]: score_field})
        else:
            raise KeyError(
                f"expected a score column (tried {score_field!r}, found no unambiguous "
                f"fallback among {list(df.columns)})"
            )
    return df.sort_values(rank_field).reset_index(drop=True)

def ranking_is_degenerate(df, score_field):
    return bool(df[score_field].nunique(dropna=False) <= 1)


## Build the condition list

In [ ]:
conditions = []
for label, model_id in MODEL_IDS.items():
    conditions.append({
        "condition_type": "model",
        "condition_label": f"model_{label}",
        "model_id": model_id,
        "prompt_label": "base",
        "raw_query": BASE_RAW_QUERY,
    })
for label, prompt_text in PROMPT_VARIANTS.items():
    conditions.append({
        "condition_type": "prompt",
        "condition_label": f"prompt_{label}",
        "model_id": None,  # env default - the same model phase 4 used
        "prompt_label": label,
        "raw_query": prompt_text,
    })

for c in conditions:
    (PILOT_DIR / c["condition_label"]).mkdir(parents=True, exist_ok=True)

print(f"{len(conditions)} conditions:")
for c in conditions:
    print(f"  {c['condition_label']:<28} model={c['model_id'] or '(env default)'}")


## Smoke test — ONE call per new condition first

Same discipline as phase 4: one call per condition here costs far less
than finding out a model-id string was wrong only after spending on the
full `N_PILOT` loop. Read every line before continuing.

In [ ]:
smoke_results = {}
for c in conditions:
    label = c["condition_label"]
    t0 = time.monotonic()
    try:
        client = OpenAICompatibleLLMClient(model=c["model_id"]) if c["model_id"] else OpenAICompatibleLLMClient()
        gen = LLMQuerySpecGenerator(client, temperature=TEMPERATURE)
        spec = gen.generate(c["raw_query"], system_hints=EXTENDED_SYSTEM_HINTS)
        spec = normalize_llm_query_spec_for_planning(spec)
        result = run_pipeline(spec)
        if result.success:
            score_field, _ = plan_score_and_rank_fields(spec)
            ranking_df = extract_ranking(result, spec)
            degenerate = ranking_is_degenerate(ranking_df, score_field)
            smoke_results[label] = "OK (degenerate ranking!)" if degenerate else "OK"
        else:
            smoke_results[label] = f"execution failed: {getattr(result, 'error', None) or getattr(result, 'structured_error', None)}"
    except Exception as e:  # noqa: BLE001 - an unfamiliar model/prompt can fail in many ways here; record, don't crash
        smoke_results[label] = f"FAILED: {type(e).__name__}: {e}"
    print(f"{label} ({time.monotonic() - t0:.1f}s): {smoke_results[label]}")

n_failed = sum(1 for v in smoke_results.values() if v.startswith("FAILED") or "execution failed" in v)
print()
print(f"{len(smoke_results) - n_failed}/{len(smoke_results)} conditions smoke-tested clean")
if n_failed:
    print("Fix the FAILED conditions above (wrong model id, or a model that rejects response_format=json_object, "
          "are the two most likely causes) before running the full loop below - or drop them from MODEL_IDS/"
          "PROMPT_VARIANTS and re-run this smoke-test cell.")


**Stop here and check before continuing.** Every condition above should
say `OK`. If any say `FAILED` or `execution failed`, fix `MODEL_IDS` (most
likely: an id string AvalAI doesn't recognize, or a model that doesn't
support `response_format={"type": "json_object"}`) or drop that condition,
then re-run the smoke test — don't spend the full `N_PILOT` loop on a
condition that's broken here.

## Full pilot loop (N_PILOT runs per condition)

In [ ]:
all_records = []

for c in conditions:
    label = c["condition_label"]
    client = OpenAICompatibleLLMClient(model=c["model_id"]) if c["model_id"] else OpenAICompatibleLLMClient()
    generator = LLMQuerySpecGenerator(client, temperature=TEMPERATURE)

    for i in range(N_PILOT):
        t0 = time.monotonic()
        record = {
            "condition_type": c["condition_type"], "condition_label": label,
            "model_id": c["model_id"], "prompt_label": c["prompt_label"],
            "run_index": i, "generation_success": False, "execution_success": False,
            "degenerate_ranking": None, "error": None,
        }
        try:
            spec = generator.generate(c["raw_query"], system_hints=EXTENDED_SYSTEM_HINTS)
            spec = normalize_llm_query_spec_for_planning(spec)
            record["generation_success"] = True
            record["query_spec"] = query_spec_to_dict(spec)
            record["operation_sequence"] = [op.name if hasattr(op, "name") else str(op) for op in spec.operations]
        except Exception as e:  # noqa: BLE001 - unfamiliar model/prompt combinations can fail in ways LLMSpecGenerationError alone doesn't cover
            record["error"] = f"generation: {type(e).__name__}: {e}"
            record["latency_s"] = time.monotonic() - t0
            all_records.append(record)
            # Write the file even on a generation failure (the first pilot run didn't -
            # a generation-only failure was silently dropped to memory-only, and lost
            # entirely when that run's kernel was interrupted before the manifest save).
            (PILOT_DIR / label / f"run_{i:02d}.json").write_text(json.dumps(record, indent=2))
            print(f"{label} run {i}: generation FAILED - {e}")
            continue

        try:
            result = run_pipeline(spec)
            record["execution_success"] = bool(result.success)
            if result.success:
                ranking_df = extract_ranking(result, spec)
                record["ranking"] = ranking_df.to_dict(orient="records")
                score_field, _ = plan_score_and_rank_fields(spec)
                record["degenerate_ranking"] = ranking_is_degenerate(ranking_df, score_field)
            else:
                record["error"] = f"execution: {getattr(result, 'error', None) or getattr(result, 'structured_error', None)}"
        except Exception as e:  # noqa: BLE001 - record, don't crash the loop
            record["error"] = f"execution: {type(e).__name__}: {e}"
            # run_pipeline() can report success (the DAG executed with no internal error)
            # even though extract_ranking() then fails on the terminal output shape - not
            # usable for Layer 3 either way, so don't let a half-set execution_success=True
            # leak into condition_metrics()'s exec_ok_clean filter downstream.
            record["execution_success"] = False
            # The first pilot run hit this exact case twice (prompt_formal, prompt_distractor):
            # the model's plan ended in a report/PDF-generation step instead of a plain ranked
            # vector. That's a genuine STRUCTURAL finding (the prompt changed the plan's shape
            # enough to add an unrequested operation), not an extraction bug - tag it as such
            # rather than burying it in a generic error string.
            if "VectorOut-like object" in str(e):
                record["structural_deviation"] = "terminal output is not a ranked vector (e.g. a report/PDF-generation step was added) - see query_spec/operation_sequence"

        record["latency_s"] = time.monotonic() - t0
        all_records.append(record)

        (PILOT_DIR / label / f"run_{i:02d}.json").write_text(json.dumps(record, indent=2))
        print(f"{label} run {i}: generation={record['generation_success']} execution={record['execution_success']} "
              f"degenerate={record['degenerate_ranking']} latency={record['latency_s']:.1f}s"
              + (f" [{record['structural_deviation']}]" if record.get("structural_deviation") else ""))


## Save the combined manifest

**Reads every condition's records from disk (`run_*.json`), not from
`all_records` in memory.** A second real run of this notebook (2026-09-16)
was interrupted partway through the full loop after the user swapped in
new model ids for the two still-broken slots — at that point `all_records`
only held entries for the conditions the loop had reached *this* kernel
session, so building the manifest/metrics from it alone would have quietly
overwritten the previous, complete results for the six untouched
conditions with blanks, even though their `run_*.json` files were still
sitting on disk untouched. Reading from disk instead makes this cell (and
the metrics cell below) correct regardless of whether the loop above ran
to completion, was interrupted, or was only re-run for a subset of
conditions.

In [ ]:
def load_condition_records(label: str) -> list[dict]:
    """Load every run_*.json this condition has on disk right now - the
    source of truth, since the full loop above may have been interrupted
    or only re-run for a subset of conditions in this kernel session."""
    return [json.loads(f.read_text()) for f in sorted((PILOT_DIR / label).glob("run_*.json"))]

manifest = pd.DataFrame([
    {
        "condition_type": r["condition_type"], "condition_label": r["condition_label"],
        "model_id": r["model_id"], "prompt_label": r["prompt_label"], "run_index": r["run_index"],
        "generation_success": r["generation_success"], "execution_success": r["execution_success"],
        "degenerate_ranking": r.get("degenerate_ranking"), "latency_s": r["latency_s"],
        "n_operations": len(r.get("operation_sequence", [])), "error": r["error"],
        "structural_deviation": r.get("structural_deviation"),
    }
    for c in conditions
    for r in load_condition_records(c["condition_label"])
])
manifest.to_csv(PILOT_DIR / "manifest.csv", index=False)
manifest


## Per-condition metrics (PAR / Rank Stability / rho vs. rule-based / reliability)

Reuses phase 5's exact `op_name_sequence`/`parse_op`/Spearman logic
(`notebooks/04_comparison_metric.ipynb`), applied per condition instead of
to one fixed arm, plus the two existing N=20 baseline rows (same model,
same prompt as phase 4 — not re-run) for a complete side-by-side table.

In [ ]:
def parse_op(s: str) -> dict:
    import re
    m = re.match(r"OperationSpec\((.*)\)$", s)
    if not m:
        raise ValueError(f"not an OperationSpec repr: {s!r}")
    ns: dict = {}
    exec(f"d = dict({m.group(1)})", {}, ns)
    return ns["d"]

def op_name_sequence(operation_sequence):
    return tuple(parse_op(s)["op"] for s in operation_sequence)

site_names = sorted(rule_based_ranking_df["name"].tolist())
rb_vector = np.array([
    dict(zip(rule_based_ranking_df["name"], rule_based_ranking_df["accessibility_score"]))[n]
    for n in site_names
], dtype=float)

def score_vector(ranking_records):
    name_to_score = {d["name"]: d["score"] for d in ranking_records}
    missing = [n for n in site_names if n not in name_to_score]
    if missing:
        raise ValueError(f"missing districts: {missing}")
    return np.array([name_to_score[n] for n in site_names], dtype=float)

def condition_metrics(label, records, n_runs):
    gen_ok = [r for r in records if r["generation_success"]]
    exec_ok_clean = [r for r in gen_ok if r["execution_success"] and not r.get("degenerate_ranking")]

    par = None
    if gen_ok:
        seqs = [op_name_sequence(r["operation_sequence"]) for r in gen_ok]
        counts = {}
        for s in seqs:
            counts[s] = counts.get(s, 0) + 1
        par = max(counts.values()) / len(seqs)

    rs_mean = rs_sd = rho_mean = rho_sd = None
    if len(exec_ok_clean) >= 1:
        vectors = [score_vector(r["ranking"]) for r in exec_ok_clean]
        rho_vs_rb = [spearmanr(v, rb_vector)[0] for v in vectors]
        rho_mean, rho_sd = statistics.mean(rho_vs_rb), (statistics.pstdev(rho_vs_rb) if len(rho_vs_rb) > 1 else 0.0)
    if len(exec_ok_clean) >= 2:
        vectors = [score_vector(r["ranking"]) for r in exec_ok_clean]
        pair_rhos = [spearmanr(vectors[i], vectors[j])[0]
                     for i in range(len(vectors)) for j in range(i + 1, len(vectors))]
        rs_mean, rs_sd = statistics.mean(pair_rhos), statistics.pstdev(pair_rhos) if len(pair_rhos) > 1 else 0.0

    latencies = [r["latency_s"] for r in records]
    return {
        "condition_label": label, "n_runs": n_runs,
        "par": par,
        "rank_stability_mean": rs_mean, "rank_stability_sd": rs_sd,
        "rho_vs_rule_based_mean": rho_mean, "rho_vs_rule_based_sd": rho_sd,
        "success_rate": (sum(1 for r in records if r["generation_success"] and r["execution_success"]) / n_runs) if n_runs else None,
        "non_degenerate_rate": len(exec_ok_clean) / n_runs if n_runs else None,
        "median_latency_s": statistics.median(latencies) if latencies else None,
    }

metrics_rows = []
for c in conditions:
    label = c["condition_label"]
    records = load_condition_records(label)  # from disk, same reasoning as the manifest cell above
    metrics_rows.append(condition_metrics(label, records, len(records) if records else N_PILOT))

# Existing N=20 baseline rows - same model/prompt as phase 4, not re-run here.
metrics_rows.append(condition_metrics("model_baseline (phase 4, N=20)", baseline_llm_runs, 20))
metrics_rows.append(condition_metrics("prompt_base (phase 4, N=20)", baseline_llm_runs, 20))

metrics_pilot = pd.DataFrame(metrics_rows)
metrics_pilot.to_csv(PILOT_DIR / "metrics_pilot.csv", index=False)
metrics_pilot


## Escalation flags

In [ ]:
def escalation_flag(row):
    if row["n_runs"] >= 20:
        return "already at N=20 (phase 4 reference)"
    if row["success_rate"] is not None and row["success_rate"] < 1.0:
        return "ESCALATE to N=20 (success_rate < 1.0)"
    if row["par"] is not None and row["par"] < 1.0:
        return "ESCALATE to N=20 (PAR < 1.0)"
    if row["rank_stability_mean"] is not None and row["rank_stability_mean"] < 1.0:
        return "ESCALATE to N=20 (Rank Stability < 1.0)"
    return "stable at N=5 pilot (not yet confirmed at N=20)"

metrics_pilot["flag"] = metrics_pilot.apply(escalation_flag, axis=1)
metrics_pilot[["condition_label", "n_runs", "par", "rank_stability_mean", "rho_vs_rule_based_mean", "success_rate", "flag"]]


## Final checks

In [ ]:
check = pd.read_csv(PILOT_DIR / "manifest.csv")
expected_rows = len(conditions) * N_PILOT
assert len(check) == expected_rows, f"expected {expected_rows} run records, found {len(check)}"
n_saved_files = sum(1 for c in conditions for _ in (PILOT_DIR / c["condition_label"]).glob("run_*.json"))
assert n_saved_files == expected_rows, f"expected {expected_rows} run_*.json files, found {n_saved_files}"
print(f"all checks passed - {len(check)} pilot runs recorded across {len(conditions)} conditions")
print()
print(metrics_pilot.to_string(index=False))


## Status: final (2026-09-16)

This pilot is complete and closed, not left open for further model
substitutions: `model_weak_legacy` and `model_frontier_other_vendor`
remain blocked (see the findings cell above) and no further AvalAI model
ids are available to try in their place. The 7 clean conditions are the
pilot's result; the 2 blocked conditions are reported as a named,
honest gap rather than resolved by more guessing. Optional future work,
not required for this pilot: escalate any of the 7 clean conditions to
`N_PILOT = 20` for tighter statistical power, or revisit the 2 blocked
slots if/when a working small or open-weight model id becomes available
on AvalAI.